# Data Ingestion Overview — How Each Source Was Pulled

This notebook is a plain-English map of where E_macro's six data pillars come
from and how each one got from a public source into `data/source_*.parquet`.
It is not a re-derivation of results (see the per-source `*_key_findings.ipynb`
notebooks and `analysis-output/source-a-findings.md` for that) — it's the
"where did this number come from" reference.

Each section covers one source: what it is, why it's in the project, how the
ingest script fetches and cleans it, any real-world snag that shaped the
final approach, and a live preview of the actual output file.

Ingest scripts live in `scripts/ingest_source_{a..f}.py`.


In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis-output" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"


## Source A — Wikipedia Intro Text (`ingest_source_a.py`)

**What it is:** The introductory paragraph(s) of each U.S. county's Wikipedia
article — the summary text before the infobox/body sections/citations.

**Why it's here:** It's the raw material for E_macro's text-embedding pillar —
a `bge-m3` embedding of what Wikipedia says about a place, used as a proxy for
"how a county is described/perceived" alongside the hard economic pillars.

**How it's pulled:** Queries the official Wikimedia Enterprise API (requires
`WIKIMEDIA_USERNAME`/`WIKIMEDIA_PASSWORD`), fetches each county's article,
and strips everything except the lead section using `BeautifulSoup` — no
infobox, no "History" or "Geography" sections, no references.

**Snag worth knowing:** 3,144 counties don't map 1:1 onto Wikipedia titles.
37 Virginia independent cities and 19 other Census-vs-Wikipedia title
mismatches needed manual backfilling (`scripts/backfill_virginia_cities.py`,
`scripts/backfill_remaining_19.py`) to reach full coverage.

**Output:** `data/source_a_text_features.parquet` (raw text + character count)
and `data/source_a_embeddings.parquet` (the 1,024-dim `bge-m3` vectors).


In [2]:
df_a = pd.read_parquet(DATA_DIR / "source_a_text_features.parquet")
print(f"{len(df_a):,} counties")
df_a[["county_name", "fips_code", "content_length"]].head()


3,144 counties


,county_name,fips_code,content_length
0,"Autauga County, Alabama",01001,85
1,"Baldwin County, Alabama",01003,713
2,"Barbour County, Alabama",01005,178
3,"Bibb County, Alabama",01007,748
4,"Blount County, Alabama",01009,577


## Source B — BLS Industrial Employment Mix (`ingest_source_b.py`)

**What it is:** BLS Quarterly Census of Employment and Wages (QCEW) —
county-level Location Quotients (LQ) for the 20 major private-sector
industries, i.e. how over/under-represented each industry is in a county
relative to the national average.

**Why it's here:** This is the "what kind of economy does this county run
on" pillar — manufacturing-heavy vs. finance-heavy vs. agriculture-heavy.

**How it's pulled:** Downloads BLS's single bulk file
(`{year}_qtrly_singlefile.zip`, 2025 Q4) rather than per-industry slices —
3 of the 20 combined NAICS codes (31-33, 44-45, 48-49) 404 as individual
slice URLs, so the bulk file is filtered locally instead. Filtered down to
private ownership, county-level rows, then pivoted long-format
(county × sector) into one row per county.

**Snag worth knowing:** BLS suppresses small-employer cells for privacy
(`disclosure_code == "N"`). Several proxy strategies (state-level fallback,
proportional allocation) were tested and none meaningfully beat a plain
null — so suppressed cells are left `NaN` with a matching `disclosure_*`
flag column, rather than guessed at.

**Output:** `data/source_b_qcew.parquet` — one `lq_emp_{naics2}` +
`disclosure_{naics2}` pair per sector.


In [3]:
df_b = pd.read_parquet(DATA_DIR / "source_b_qcew.parquet")
print(f"{len(df_b):,} counties, {len(df_b.columns) - 2} columns")
df_b[["county_name", "fips_code", "lq_emp_31-33", "disclosure_31-33"]].head()


3,143 counties, 40 columns


,county_name,fips_code,lq_emp_31-33,disclosure_31-33
0,"Autauga County, Alabama",01001,1.86,False
1,"Baldwin County, Alabama",01003,0.60,False
2,"Barbour County, Alabama",01005,3.46,False
3,"Bibb County, Alabama",01007,1.56,False
4,"Blount County, Alabama",01009,1.84,False


## Source C — FRED Unemployment & GDP Trends (`ingest_source_c.py`)

**What it is:** County-level annual unemployment rate and real GDP from the
Federal Reserve's FRED database, converted into a 3-year rolling first
derivative — i.e. is the county's economy accelerating or decelerating,
not just what its level is.

**Why it's here:** The trend/momentum pillar — a snapshot LQ or income ratio
says where a county stands; this says which direction it's moving.

**How it's pulled:** FRED API, annual frequency only. Monthly county
unemployment series use ad-hoc state/county abbreviation codes that don't
map cleanly to a FIPS code, while the annual series are directly
FIPS-derivable (`LAUCN{FIPS}0000000003A`, `REALGDPALL{FIPS}`) — so annual was
chosen deliberately, not for lack of monthly data. Requires `FRED_API_KEY`.

**Output:** `data/source_c_fred.parquet` — `unemployment_velocity` and
`gdp_velocity` (the 3-year slopes) plus latest-year levels for context.


In [4]:
df_c = pd.read_parquet(DATA_DIR / "source_c_fred.parquet")
print(f"{len(df_c):,} counties")
df_c[["county_name", "fips_code", "unemployment_velocity", "gdp_velocity"]].head()


3,144 counties


,county_name,fips_code,unemployment_velocity,gdp_velocity
0,"Autauga County, Alabama",01001,0.066667,6598.333333
1,"Baldwin County, Alabama",01003,0.100000,503735.666667
2,"Barbour County, Alabama",01005,0.000000,-3104.333333
3,"Bibb County, Alabama",01007,0.100000,5115.666667
4,"Blount County, Alabama",01009,0.100000,47252.666667


## Source D — Freight Flows (`ingest_source_d.py`)

**What it is:** BTS Freight Analysis Framework (FAF5) county-level freight
tonnage — how much a county ships/receives by commodity type, and how
concentrated its trading partners are (an HHI concentration index).

**Why it's here:** The logistics/trade pillar — is this county a shipping
hub with diverse partners, or dependent on one or two trade corridors.

**How it's pulled:** Per-state zip files from `faf.ornl.gov`, each containing
origin-destination tables. Only a county's own "home" state zip is used
(not its appearance as a neighbor in another state's zip), since only the
home file guarantees the complete adjacent-neighbor set.

**Snag worth knowing:** The BTS landing page blocks plain HTTP clients
(bot detection), but the actual data host doesn't — except that host resets
the connection mid-handshake specifically against Python's `ssl`/`urllib3`
stack, while system `curl` connects fine. The script shells out to `curl`
for downloads rather than fighting the Python TLS stack.

**Output:** `data/source_d_faf.parquet` — inbound/outbound tonnage totals,
partner-concentration HHI, and per-commodity-group tonnage splits.


In [5]:
df_d = pd.read_parquet(DATA_DIR / "source_d_faf.parquet")
print(f"{len(df_d):,} counties")
df_d[["county_name", "fips_code", "total_outbound_tons", "total_inbound_tons", "out_partner_hhi"]].head()


3,144 counties


,county_name,fips_code,total_outbound_tons,total_inbound_tons,out_partner_hhi
0,"Autauga County, Alabama",01001,2132.708707,3989.310310,0.017767
1,"Baldwin County, Alabama",01003,7142.368661,12752.046050,0.149774
2,"Barbour County, Alabama",01005,1588.375946,1349.542598,0.015944
3,"Bibb County, Alabama",01007,1246.223770,2801.294216,0.055737
4,"Blount County, Alabama",01009,1491.392670,3520.044184,0.068469


## Source E — IRS Capital Composition (`ingest_source_e.py`)

**What it is:** IRS Statistics of Income (SOI) county file, Tax Year 2022 —
the ratio of investment income (capital gains + qualified dividends) to
W-2 wage income per county.

**Why it's here:** A wealth-composition signal — is county income mostly
paycheck-driven or investment-driven, which behaves very differently than
the raw employment/GDP numbers in Sources B/C.

**How it's pulled:** Downloads the IRS's own pre-aggregated county totals
file (`22incyallnoagi.csv`) directly — no API key needed, no bot protection
on the IRS file host. Target columns are looked up by name via a
`SOI_COLUMN_MAP` rather than positional index, so a future IRS schema change
fails loudly instead of silently misreading a shifted column.

**Snag worth knowing:** Unlike BLS's `disclosure_code`, the IRS file has no
suppression flag. A handful of very low-population counties show exact-zero
amounts that could be genuine or a hidden small-cell suppression — both are
written identically, since there's no way to tell them apart from this file.

**Output:** `data/source_e_irs_soi.parquet` — `capital_to_wage_ratio` plus
the underlying income components and a `low_return_flag` for thin-sample
counties.


In [6]:
df_e = pd.read_parquet(DATA_DIR / "source_e_irs_soi.parquet")
print(f"{len(df_e):,} counties")
df_e[["county_name", "fips_code", "capital_to_wage_ratio", "low_return_flag"]].head()


3,143 counties


,county_name,fips_code,capital_to_wage_ratio,low_return_flag
0,"Autauga County, Alabama",01001,0.040277,False
1,"Baldwin County, Alabama",01003,0.170460,False
2,"Barbour County, Alabama",01005,0.079666,False
3,"Bibb County, Alabama",01007,0.024429,False
4,"Blount County, Alabama",01009,0.039927,False


## Source F — USDA County Typology (`ingest_source_f.py`)

**What it is:** USDA Economic Research Service's 2025 County Typology Codes —
a structural classification of each county (farming-dependent,
manufacturing-dependent, recreation-dependent, persistent poverty, population
loss, etc.), one-hot encoded.

**Why it's here:** A "baseline anchor" pillar — USDA's own expert
classification, refreshed on a slow annual/decennial cycle rather than a
live time series, used as a structural label to sanity-check the other five
pillars against.

**How it's pulled:** A single static CSV download from `ers.usda.gov`, no
API key or rate limiting. Pivoted from USDA's long format into one row per
county, one-hot encoding the mutually-exclusive Industry Dependence
category, then joined onto the project's FIPS crosswalk.

**Output:** `data/source_f_usda_typology.parquet` — one boolean column per
typology category (`high_farming`, `high_manufacturing`, `population_loss`,
`persistent_poverty`, etc.).


In [7]:
df_f = pd.read_parquet(DATA_DIR / "source_f_usda_typology.parquet")
print(f"{len(df_f):,} counties, {len(df_f.columns) - 2} typology flags")
df_f[["county_name", "fips_code", "high_manufacturing", "population_loss", "persistent_poverty"]].head()


3,144 counties, 19 typology flags


,county_name,fips_code,high_manufacturing,population_loss,persistent_poverty
0,"Autauga County, Alabama",01001,False,False,False
1,"Baldwin County, Alabama",01003,False,False,False
2,"Barbour County, Alabama",01005,True,True,True
3,"Bibb County, Alabama",01007,False,False,False
4,"Blount County, Alabama",01009,False,False,False


## Coverage Summary

All six sources target the same 3,144 U.S. counties (via the shared FIPS
crosswalk in `data/county_crosswalk.parquet`), though a few sources land at
3,143 due to one county falling out at a specific step (e.g. a BLS or IRS
suppression edge case) rather than a systemic gap. See each source's
`*_key_findings.ipynb` for what happens after ingestion — clustering,
correlation with other pillars, and cross-validation.
